In [1]:
import pandas as pd
deliveries = pd.DataFrame({
    "delivery_id": [1, 2, 3, 4, 5, 6],
    "delivered_at": [
        "2026-01-02 09:00",
        "2026-01-03 14:30",
        "2026-01-10 18:00",
        "2026-02-01 08:15",
        "invalid",
        "2026-02-15 20:00",
    ],
    "delivery_hours": [4.5, 7.0, 9.5, 3.0, 8.0, 10.0],
    "status": ["on_time", "late", "late", "on_time", "late", "late"],
})
deliveries

,delivery_id,delivered_at,delivery_hours,status
0,1,2026-01-02 09:00,4.5,on_time
1,2,2026-01-03 14:30,7.0,late
2,3,2026-01-10 18:00,9.5,late
3,4,2026-02-01 08:15,3.0,on_time
4,5,invalid,8.0,late
5,6,2026-02-15 20:00,10.0,late


In [3]:
deliveries["delivered_at"]=pd.to_datetime(
    deliveries["delivered_at"],
    errors="coerce"
    )
deliveries

,delivery_id,delivered_at,delivery_hours,status
0,1,2026-01-02 09:00:00,4.5,on_time
1,2,2026-01-03 14:30:00,7.0,late
2,3,2026-01-10 18:00:00,9.5,late
3,4,2026-02-01 08:15:00,3.0,on_time
4,5,NaT,8.0,late
5,6,2026-02-15 20:00:00,10.0,late


In [4]:
invalid_timestamp=deliveries[deliveries["delivered_at"].isna()]
print(invalid_timestamp)

   delivery_id delivered_at  delivery_hours status
4            5          NaT             8.0   late


In [5]:
deliveries=deliveries.dropna()
deliveries

,delivery_id,delivered_at,delivery_hours,status
0,1,2026-01-02 09:00:00,4.5,on_time
1,2,2026-01-03 14:30:00,7.0,late
2,3,2026-01-10 18:00:00,9.5,late
3,4,2026-02-01 08:15:00,3.0,on_time
5,6,2026-02-15 20:00:00,10.0,late


In [9]:
deliveries["month"]=deliveries["delivered_at"].dt.month_name()
deliveries["day"]=deliveries["delivered_at"].dt.day_name()
deliveries["hour"]=deliveries["delivered_at"].dt.hour
deliveries

,delivery_id,delivered_at,delivery_hours,status,month,day,hour
0,1,2026-01-02 09:00:00,4.5,on_time,January,Friday,9
1,2,2026-01-03 14:30:00,7.0,late,January,Saturday,14
2,3,2026-01-10 18:00:00,9.5,late,January,Saturday,18
3,4,2026-02-01 08:15:00,3.0,on_time,February,Sunday,8
5,6,2026-02-15 20:00:00,10.0,late,February,Sunday,20


In [14]:
monthly_avg_delivery=deliveries.groupby("month",as_index=False).agg(avg_delivery_hours=("delivery_hours","mean"))
monthly_avg_delivery

,month,avg_delivery_hours
0,February,6.5
1,January,7.0


In [15]:
deliveries["is_late"]=deliveries["status"]=="late"
deliveries

,delivery_id,delivered_at,delivery_hours,status,month,day,hour,is_late
0,1,2026-01-02 09:00:00,4.5,on_time,January,Friday,9,False
1,2,2026-01-03 14:30:00,7.0,late,January,Saturday,14,True
2,3,2026-01-10 18:00:00,9.5,late,January,Saturday,18,True
3,4,2026-02-01 08:15:00,3.0,on_time,February,Sunday,8,False
5,6,2026-02-15 20:00:00,10.0,late,February,Sunday,20,True


In [17]:
late_rate_by_weekday=(
    deliveries.groupby("day")["is_late"]
    .mean()
    .mul(100)
    .reset_index(name="late_rate_pct")
)
late_rate_by_weekday

,day,late_rate_pct
0,Friday,0.0
1,Saturday,100.0
2,Sunday,50.0


In [19]:
deliveries["is_evening"]=deliveries["hour"]>=18
deliveries

,delivery_id,delivered_at,delivery_hours,status,month,day,hour,is_late,is_evenining,is_evening
0,1,2026-01-02 09:00:00,4.5,on_time,January,Friday,9,False,False,False
1,2,2026-01-03 14:30:00,7.0,late,January,Saturday,14,True,False,False
2,3,2026-01-10 18:00:00,9.5,late,January,Saturday,18,True,True,True
3,4,2026-02-01 08:15:00,3.0,on_time,February,Sunday,8,False,False,False
5,6,2026-02-15 20:00:00,10.0,late,February,Sunday,20,True,True,True


In [20]:
deliveries.groupby("is_evening")["delivery_hours"].mean()

is_evening
False    4.833333
True     9.750000
Name: delivery_hours, dtype: float64

In [21]:
(deliveries["status"]=="late").groupby(deliveries["is_evening"]).mean().mul(100)

is_evening
False     33.333333
True     100.000000
Name: status, dtype: float64